In [18]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from configE import Config
from solverE import assemble_matrix, solve_system   # safe
# from solverE_jit import assemble_matrix, solve_system   # jit
from helperE import GIF_maker
from helperE import plot_3d_gif

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
M = 100

# Create configuration
cfg = Config(dt=0.001, t_final=1.0, M=M)

# Assemble and solve
C_matrix = assemble_matrix(cfg)
print(f"\nCreated {C_matrix.shape} matrix.")
print(f"Number of dimensions: {C_matrix.ndim}")
print(f"Number of non-zero elements: {C_matrix.nnz}")



Created (10201, 10201) matrix.
Number of dimensions: 2
Number of non-zero elements: 50601


In [20]:
data = {
    "Parameter": ["M", "N", "dt", "t_final", "Bi", "S", "D", "gamma", "epsilon", "m", "n", "U_threshold", "alpha_init"],
    "Value": [cfg.M, cfg.N, cfg.dt, cfg.t_final, cfg.Bi, cfg.S, cfg.D, cfg.gamma, cfg.epsilon, cfg.m, cfg.n, cfg.U_threshold, cfg.params["alpha"]],
}

df = pd.DataFrame(data)
df["Value"] = df["Value"].apply(lambda x: f"{x:.2e}")
df

,Parameter,Value
0,M,1.00e+02
1,N,1.01e+02
2,dt,1.00e-03
3,t_final,1.00e+00
4,Bi,2.00e+01
5,S,9.20e-01
6,D,2.09e+09
7,gamma,2.17e+01
8,epsilon,3.23e-01
9,m,1.20e+00


In [21]:
snap_interval = 10

history_U, history_alpha, history_alpha_rate = solve_system(cfg, C_matrix, snap_interval)

Solving system:   0%|          | 0/1000 [00:00<?, ?it/s]

In [22]:
print(f"Temperature scaling U: {history_U[0].reshape((M+1, M+1))}")
print(f"Alpha: {np.array(history_alpha[-1])}")
print(f"Alpha rate: {np.array(history_alpha_rate[5])}")

Temperature scaling U: [[-0.39904816 -0.46383399 -0.50870547 ... -0.50870547 -0.46383399
  -0.39904816]
 [-0.46383399 -0.53974967 -0.59265282 ... -0.59265282 -0.53974967
  -0.46383399]
 [-0.50870547 -0.59265282 -0.65170925 ... -0.65170925 -0.59265282
  -0.50870547]
 ...
 [-0.50870547 -0.59265282 -0.65170925 ... -0.65170925 -0.59265282
  -0.50870547]
 [-0.46383399 -0.53974967 -0.59265282 ... -0.59265282 -0.53974967
  -0.46383399]
 [-0.39904816 -0.46383399 -0.50870547 ... -0.50870547 -0.46383399
  -0.39904816]]
Alpha: [0.67043985 0.67026194 0.67009641 ... 0.67009641 0.67026194 0.67043985]
Alpha rate: [0.07172962 0.07026288 0.06882847 ... 0.06882847 0.07026288 0.07172962]


In [23]:
# Make GIF
GIF_maker(
    histories=[history_U, history_alpha],
    titles=[r"Temperature scaling $U$", r"Cure degree $\alpha$"],
    cmaps=['hot', 'viridis'],
    vmins=[-1.0, 0.0],
    vmaxs=[0.0, 1.0],
    filename='rubber_heating.gif',
    N=cfg.N,
    snap_interval=snap_interval,
    dt=cfg.dt
)

GIF saved as 'rubber_heating.gif' (100 frames, 10 fps)


In [24]:


plot_3d_gif(
    histories=[history_U, history_alpha],
    titles=[r"Temperature $U$", r"Cure degree $\alpha$"],
    cmaps=['hot', 'viridis'],
    vmins=[-1.0, 0.0],
    vmaxs=[0.0, 1.0],
    filename='rubber_heating_3d.gif',
    N=cfg.N,
    snap_interval=snap_interval,
    dt=cfg.dt,
    fps=10,
    elev=30,
    azim=225
)

3D GIF saved as 'rubber_heating_3d.gif' (100 frames, 10 fps)
